# Play chess against ChessGPT

This notebook lets you play an interactive game against ChessGPT (432M params).

- Enter your moves in UCI notation (e.g. `e2e4`, `g1f3`, `e7e8q`)
- Type `quit` to resign

In [ ]:
import torch
import chess
import chess.svg
from IPython.display import display, SVG, clear_output

from chessgpt import resolve_device
from chessgpt.generate import (
    load_checkpoint,
    sample_next_token,
    build_legal_mask,
)

# Load model from the HuggingFace Hub (or a local path)
CHECKPOINT = "malcouffe/chessgpt"

device = resolve_device("auto")
model, tokenizer, config = load_checkpoint(CHECKPOINT, device)
print(f"Model loaded on {device} — {sum(p.numel() for p in model.parameters()):,} params")
print(f"Architecture: d_model={config.d_model}, n_layers={config.n_layers}, n_heads={config.n_heads}")

In [ ]:
# --- Game settings ---

PLAYER_COLOR = chess.WHITE   # chess.WHITE or chess.BLACK

# Sampling
TEMPERATURE = 0.4
TOP_K = 40
TOP_P = 0.95
GREEDY = False               # True = always pick the model's top move

In [ ]:
def model_play_move(board, history, model, tokenizer, device):
    """Have the model play a move and return the UCI string."""
    ids = [tokenizer.BOS_ID] + [tokenizer.move_to_id[m] for m in history]

    # Crop to max context length
    max_ctx = model.config.max_seq_len
    if len(ids) > max_ctx:
        ids = ids[-max_ctx:]

    input_ids = torch.tensor([ids], dtype=torch.long, device=device)

    legal_mask = build_legal_mask(board, tokenizer, device, allow_eos=False)

    tid = sample_next_token(
        model=model,
        input_ids=input_ids,
        tokenizer=tokenizer,
        legal_mask=legal_mask,
        temperature=TEMPERATURE,
        top_k=TOP_K,
        top_p=TOP_P,
        greedy=GREEDY,
    )

    move_uci = tokenizer.id_to_move.get(tid, "")
    if not move_uci or chess.Move.from_uci(move_uci) not in board.legal_moves:
        # Fallback (should not happen with legal_mask)
        import random
        mv = random.choice(list(board.legal_moves))
        move_uci = mv.uci()

    return move_uci


def show_board(board, last_move=None):
    """Display the board as SVG."""
    flipped = (PLAYER_COLOR == chess.BLACK)
    arrows = []
    if last_move is not None:
        arrows = [chess.svg.Arrow(last_move.from_square, last_move.to_square, color="#88cc44")]
    svg = chess.svg.board(board, flipped=flipped, arrows=arrows, size=400)
    display(SVG(svg))

In [ ]:
board = chess.Board()
history = []        # list of UCI moves
last_move = None
move_number = 1

print("Game started! You play", "White" if PLAYER_COLOR == chess.WHITE else "Black")
print("Enter moves in UCI notation (e.g. e2e4). Type 'quit' to resign.\n")
show_board(board)

while not board.is_game_over():
    is_player_turn = (board.turn == PLAYER_COLOR)

    if is_player_turn:
        # Player's turn
        while True:
            user_input = input(f"\nMove {move_number}{'. ' if board.turn == chess.WHITE else '... '}: ").strip()
            if user_input.lower() == "quit":
                print("\nYou resigned. Game over.")
                board = None
                break
            try:
                mv = chess.Move.from_uci(user_input)
                if mv not in board.legal_moves:
                    print(f"  Illegal move. Legal moves: {' '.join(m.uci() for m in board.legal_moves)}")
                    continue
                break
            except ValueError:
                print(f"  Invalid format. Use UCI notation (e.g. e2e4, g1f3)")

        if board is None:
            break

        san = board.san(mv)
        move_uci = mv.uci()
    else:
        # Model's turn
        move_uci = model_play_move(board, history, model, tokenizer, device)
        mv = chess.Move.from_uci(move_uci)
        san = board.san(mv)

    # Play the move
    board.push(mv)
    history.append(move_uci)
    last_move = mv

    # Display
    clear_output(wait=True)
    who = "You" if is_player_turn else "ChessGPT"
    prefix = f"{move_number}. " if board.turn == chess.BLACK else f"{move_number}... "
    print(f"{who} played: {prefix}{san} ({move_uci})\n")

    # Move counter
    if board.turn == chess.WHITE:
        move_number += 1

    show_board(board, last_move)

# End of game
if board is not None and board.is_game_over():
    outcome = board.outcome()
    print(f"\nGame over! Result: {board.result()}")
    if outcome.termination == chess.Termination.CHECKMATE:
        winner = "White" if outcome.winner == chess.WHITE else "Black"
        print(f"Checkmate — {winner} wins!")
    elif outcome.termination == chess.Termination.STALEMATE:
        print("Stalemate — draw.")
    else:
        print(f"Draw ({outcome.termination.name}).")